In [2]:
import pandas as pd

In [3]:
all_oligos = pd.read_excel('notebooks/jacob/data/20250309_FolDE_primers.xlsx')

In [4]:
long_oligos = all_oligos[all_oligos.length > 60].copy()

In [10]:
for index, row in long_oligos.iterrows():
    print(f"{row['Name']}\t{row['sequence']}\t100nm")

GAH_DBAT.M417L.F1	CCCAGATGGGATAAAGATTTTGTCATTCCTACCTCCACTGATAATGAAAAGTTTCAAATTCGAG	100nm
GAH_DBAT.G227D.F1	CGAGTACCTTTGGCAAAATAGTTCAAGATTCCTTAGTCATCACGAGTGAAACCATTAACTG	100nm
GAH_DBAT.C129K.F1	CTCTCCTAGCTTGGAACAATTGTTATTTAAACTACCGCCGGATACAGATATAGAAGACATC	100nm
GAH_DBAT.Q66E.F1	GACCCTGCAAAGGTGATCCGTGAGGCTTTGTCCAAAGTATTAGTCTACTATTCCCCATTTG	100nm
GAH_DBAT.L294P.F1	GATATGCGTAAATTATTTAATCCCCCCTTACCAAAAGGATATTATGGTAACTTCGTGGGAACAGTTTG	100nm
GAH_DBAT.G227E.F1	CGAGTACCTTTGGCAAAATAGTTCAAGAGTCCTTAGTCATCACGAGTGAAACCATTAACTG	100nm
PW_CYP90.Q87P.F1	GATATTTAAAGCGAATTTATTCGCAAGTCCTGCTGTAGTATCCGTAGATGCTGAATTGAAC	100nm
PW_CYP90.Q437E.F1	GGATTAAGAAATTGCGCCGGATTAGAATTGGCTAAGCTAGAGATAGTTGTTTTTCTACATC	100nm
PW_CYP90.V444S.F1	GATTACAATTGGCTAAGCTAGAGATATCCGTTTTTCTACATCACCTGGTGTTAAACTTCGATTG	100nm
PW_CYP90.A157I.F1	GGATGAAGTCATACTTTTTGCCTGATATCGAACGTTATATTACCGAAACTTTGGCTTCATG	100nm
PW_GABAT.W214F.F1	GTTTTGCACTCAGATTGTCCACACTACTTTAGGTACCATTTACCTGGGGAAAGTGAAGAAG	100nm
PW_GABAT.S228T.F1	CTGGGGAAAGTGAAGAAGAATTCTC

In [34]:
def number_to_well_384(number):
    """
    Convert a number (0-383) to a 384-well plate position (A1-P24).
    
    Args:
        number (int): Number between 0 and 383
        
    Returns:
        str: Well position (e.g., 'A1', 'B2', etc.)
        
    Raises:
        ValueError: If number is outside valid range
    """
    if not 0 <= number <= 383:
        raise ValueError("Number must be between 0 and 383")
    
    # 384-well plate has 16 rows (A-P) and 24 columns (1-24)
    row = number // 24  # Integer division to get row number
    col = number % 24   # Remainder to get column number
    
    # Convert row number to letter (0=A, 1=B, etc.)
    row_letter = chr(65 + row)  # 65 is ASCII for 'A'
    
    # Convert column to 1-based index
    col_number = col + 1
    
    return f"{row_letter}{col_number}"

assert number_to_well_384(0) == 'A1'
assert number_to_well_384(23) == 'A24'
assert number_to_well_384(24) == 'B1'


In [37]:
oligos = all_oligos[all_oligos.length <= 60].copy()
oligos = oligos[~oligos.Name.str.startswith('ID_BezEu')]

assert oligos[oligos.Name.apply(
  lambda x: any([v in x for v in ['H162E', 'R262F', 'G243E', 'R262N']])
)].shape[0] == 0, 'Some of Mias oligos are in here...'

oligos['campaign'] = oligos.Name.str.split('.').str[0]
oligos['seq_id'] = oligos.Name.str.split('.').str[1]
oligos['locus'] = oligos.seq_id.apply(lambda x: int(x[1:-1]))
oligos['direction'] = oligos.Name.str.split('.').str[2]

oligos = oligos.sort_values(by=['campaign', 'direction', 'locus', 'seq_id'])

oligos = oligos.reset_index(drop=True)
oligos['well'] = oligos.index.map(number_to_well_384)

oligos.head(25)

,Name,sequence,length,campaign,seq_id,locus,direction,well
0,AP_OplR.W10L.F1,GAACTCTCGACCCACGGCCTGCCACAGCCAGAGCGCCAGGTAC,43,AP_OplR,W10L,10,F1,A1
1,AP_OplR.S51D.F1,GCGCTGGAGTACGCCGGACACGCCGCTCAGCCTGTCGC,38,AP_OplR,S51D,51,F1,A2
2,AP_OplR.Q63A.F1,GCGCCTGCGGTCCAGCGCCCTGGGTTACTCACGCAGCAAAGCG,43,AP_OplR,Q63A,63,F1,A3
3,AP_OplR.Q63G.F1,GCGCCTGCGGTCCAGCGGCCTGGGTTACTCACGCAGCAAAGCG,43,AP_OplR,Q63G,63,F1,A4
4,AP_OplR.F80A.F1,CGGCCAGGACCATGAAGCGGCCTACCTGGTGACCGTGCCGCATGG,45,AP_OplR,F80A,80,F1,A5
5,AP_OplR.F80C.F1,CGGCCAGGACCATGAAGCGTGCTACCTGGTGACCGTGCCGCATGG,45,AP_OplR,F80C,80,F1,A6
6,AP_OplR.F80S.F1,CGGCCAGGACCATGAAGCGAGCTACCTGGTGACCGTGCCGCATGG,45,AP_OplR,F80S,80,F1,A7
7,AP_OplR.Q123D.F1,CGCTTCCACTACGGCACGGACAACGACTTGTGGGTGCTCAAGCTG,45,AP_OplR,Q123D,123,F1,A8
8,AP_OplR.A135W.F1,CTCAAGCTGCCCGAGCGTTGGCTCAAAGCCAACCTGCTGGGGCAC,45,AP_OplR,A135W,135,F1,A9
9,AP_OplR.N139A.F1,GAGCGTGCGCTCAAAGCCGCCCTGCTGGGGCACAAGCGCTATAC,44,AP_OplR,N139A,139,F1,A10


In [40]:
oligos.rename(columns={
  'Name': 'Name',
  'well': 'Well Position',
  'sequence': 'Sequence'
})[['Well Position', 'Name', 'Sequence']].to_excel('notebooks/jacob/data/250310_ordered_primers.xlsx', index=False)